# Сравнение восстановления фазы — без обучения

Один предобученный **MP-SENet** предсказывает амплитуду и фазу. Сравниваем:

| Метод | Амплитуда до ISTFT | Фаза |
|---|---|---|
| `noisy` | Исходная | Исходная (baseline) |
| `original_phase` | Предсказанная MP-SENet | Исходная зашумлённая |
| `griffin_lim` | Та же предсказанная | Итерации Griffin–Lim |
| `mp_senet` | Та же предсказанная | Предсказанная MP-SENet |

**Вопрос эксперимента:** как выбор фазы влияет на результат при фиксированной предсказанной амплитуде.
Амплитуда получена моделью, обученной совместно с фазовой веткой; это не сравнение трёх отдельно обученных систем.
После ISTFT и повторной STFT фактические амплитуды могут различаться из-за согласованности спектра.

Запуск: Jupyter / Kaggle / Colab, Python 3.10–3.12 рекомендуется. GPU желателен, CPU поддерживается.
Нужны интернет для пакетов и официальных весов, а также **свои тестовые пары noisy/clean**.
Загрузка данных и обучение здесь отсутствуют. Сначала запустите 20 файлов, затем `N_FILES = None` для всего теста.

Используется фиксированный официальный checkpoint `g_best_vb` (VoiceBank+DEMAND).
Это публичная версия репозитория, а не заявление о точном воспроизведении статьи 2023 года.
Не меняем раскладку осей attention или код авторов после загрузки весов.
Используйте официальный test split, не обучающие файлы; проверить происхождение ваших файлов автоматически нельзя.

Источники: [MP-SENet и веса](https://github.com/yxlu-0102/MP-SENet),
[статья 2023](https://www.isca-archive.org/interspeech_2023/lu23e_interspeech.pdf).


## 1. Установка
При первом запуске выполняется установка зависимостей. Если обновлялся уже импортированный пакет, перезапустите kernel.
Обучающие зависимости из старого `requirements.txt` авторов не устанавливаются.


In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy>=1.24,<3", "setuptools", "wheel", "Cython"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "torch>=2.1,<3", "scipy>=1.10,<2", "pandas>=2,<3", "matplotlib>=3.7,<4",
    "soundfile>=0.12,<1", "joblib>=1.3,<2", "pystoi==0.4.1",
    "tqdm>=4.65,<5", "ipython>=8,<10",
])
subprocess.check_call([sys.executable, "-m", "pip", "install", "pesq==0.0.4", "--no-build-isolation"])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.0/639.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5
  Attempting uninstall: ipython
    Found existing installation: ipython 7.34.0
    Uninstalling ipython-7.34.0:
      Successfully uninstalled ipython-7.34.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.17.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for pesq: filename=pesq-0.0.4-cp313-cp313-linux_x86_64.whl size=279143 sha256=31faffa3da6ba4a7f1f9444769b3cc5840c44b971dfe6bc7947a8f0538c19c79
  Stored in directory: /root/.cache/pip/wheels/7f/fd/bd/0db325a6b4eee46eb45834f881c452d8b651e8ee8dc86546f9
Successfully built pesq


0

## 2. Пути и параметры
Измените `NOISY_DIR` и `CLEAN_DIR`: это директории WAV с одинаковыми именами файлов.
По умолчанию указаны пути из вашего предыдущего Kaggle-ноутбука.
Для теста целиком установите `N_FILES = None`. Параметры Griffin–Lim фиксируйте до итогового теста.


In [2]:
from pathlib import Path

NOISY_DIR = Path("/kaggle/input/datasets/jweiqi/voicebank-demand-16k/noisy_testset_wav")
CLEAN_DIR = Path("/kaggle/input/datasets/jweiqi/voicebank-demand-16k/clean_testset_wav")
WORK_DIR = (Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()) / "mp_senet_phase_results"

N_FILES = 20                 # None — весь набор; подвыборка случайная, но воспроизводимая
SEED = 1234
GL_ITERATIONS = 60
GL_INITIAL_PHASE = "noisy"   # "noisy" или "random"; в обоих случаях далее идёт Griffin–Lim
DEVICE = "auto"             # "auto", "cpu", "cuda"; auto выбирает CUDA или CPU
SAVE_AUDIO_EXAMPLES = 5      # сохраняем первые N выбранных записей для прослушивания
BOOTSTRAP_REPEATS = 2000


In [3]:
import hashlib
import importlib
import importlib.metadata
import json
import math
import platform
import random
import sys
import time
import urllib.request
import uuid
import warnings
from datetime import datetime, timezone
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import torch
from scipy.signal import resample_poly
from pesq import pesq
from pystoi import stoi
from tqdm.auto import tqdm
from IPython.display import Audio, display

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if DEVICE == "auto" and torch.cuda.is_available() else ("cpu" if DEVICE == "auto" else DEVICE))
if device.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA недоступна. Выберите DEVICE='cpu' или включите GPU.")
if device.type not in {"cpu", "cuda"}:
    raise ValueError("Для этого эксперимента используйте CPU или CUDA.")
if GL_INITIAL_PHASE not in {"noisy", "random"} or GL_ITERATIONS < 1:
    raise ValueError("Проверьте GL_INITIAL_PHASE и GL_ITERATIONS.")
if N_FILES is not None and (not isinstance(N_FILES, int) or N_FILES < 1):
    raise ValueError("N_FILES должен быть положительным целым или None.")

WORK_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = WORK_DIR / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6])
RUN_DIR.mkdir()
REPO_DIR = WORK_DIR / "official_mp_senet"
METHODS = ["noisy", "original_phase", "griffin_lim", "mp_senet"]
METRICS = ["PESQ", "ESTOI", "SI_SDR"]
print("Device:", device, "| torch:", torch.__version__)
print("Результаты:", RUN_DIR)


Device: cpu | torch: 2.11.0+cpu
Результаты: /kaggle/working/mp_senet_phase_results/20261016T142012Z_e47b04


## 3. Официальный код и checkpoint
Загружаются только нужные файлы с фиксированного commit; каждый проверяется по SHA-256.
Существующий несовпадающий файл не перезаписывается молча.
Загрузка checkpoint выполняется через `weights_only=True`, затем проверяется полное совпадение ключей.
Старый checkpoint U-Net не нужен.


In [4]:
REVISION = "89932cfe90d1dacb8e170e4a331d762462c21792"
SOURCE_HASHES = {'models/model.py': '4e138d64047cf98f9f43a288206594c9c0a060328bae738e5c3667e223bf4db1', 'models/transformer.py': 'e7f5dd6c11551be5cb1495682661da20edadb221c91c57d363f686a74cdd9941', 'utils.py': 'd14fda2545763e6f5485468727aadaf14837df715bc0d9a07708f99d389dc542', 'best_ckpt/config.json': '339cb60721e4ab0ee3ed4825ca9dfc981c7554c7b5e87f043e454e5a17b58f92', 'best_ckpt/g_best_vb': 'aedfb1aa549159f71b39613d94db831dfa983b3a68b0deea02e84e0ad563f4f9', 'LICENSE': '858f31052a5df6bcec94b015607bfade5a7cc6e950f7a9822aa4da3cc6f62fca'}

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def download_verified(relative, expected):
    destination = REPO_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        if file_sha256(destination) != expected:
            raise RuntimeError(f"Контрольная сумма не совпала: {destination}. Удалите этот файл и повторите ячейку.")
        return destination
    url = f"https://raw.githubusercontent.com/yxlu-0102/MP-SENet/{REVISION}/{relative}"
    temporary = destination.with_name(destination.name + ".part")
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "mp-senet-phase-comparison"})
        with urllib.request.urlopen(request, timeout=120) as response, temporary.open("wb") as out:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                out.write(chunk)
        if file_sha256(temporary) != expected:
            raise RuntimeError(f"Неверная контрольная сумма скачанного {relative}")
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination

for relative, expected in SOURCE_HASHES.items():
    download_verified(relative, expected)
print("Официальные код, конфигурация и веса проверены.")


Официальные код, конфигурация и веса проверены.


In [5]:
# Импорты авторов используют имена models и utils. Не смешиваем их с другими проектами kernel.
for module_name in ("utils", "models", "models.model", "models.transformer"):
    existing = sys.modules.get(module_name)
    if existing is not None:
        origin = getattr(existing, "__file__", None)
        namespace_paths = list(getattr(existing, "__path__", []))
        locations = ([origin] if origin else []) + namespace_paths
        if not locations or any(not Path(p).resolve().is_relative_to(REPO_DIR.resolve()) for p in locations):
            raise RuntimeError(f"Модуль {module_name} уже занят другим кодом. Перезапустите kernel.")
if str(REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_DIR.resolve()))
from models.model import MPNet

h = SimpleNamespace(**json.loads((REPO_DIR / "best_ckpt/config.json").read_text()))
model = MPNet(h, num_tsblocks=h.num_tsconformers).to(device)
checkpoint = torch.load(REPO_DIR / "best_ckpt/g_best_vb", map_location="cpu", weights_only=True)
model.load_state_dict(checkpoint["generator"], strict=True)
del checkpoint
model.eval()
model.requires_grad_(False)

SR = int(h.sampling_rate)
WINDOW = torch.hann_window(h.win_size, periodic=True, device=device)
print("Параметров:", f"{sum(p.numel() for p in model.parameters()):,}")
print("SR / FFT / hop / window / compression:", SR, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
print("Используется исходная реализация checkpoint, без изменения attention.")


Параметров: 2,263,372
SR / FFT / hop / window / compression: 16000 400 100 400 0.3
Используется исходная реализация checkpoint, без изменения attention.


## 4. Общие STFT / ISTFT и Griffin–Lim
Во всех ветках одинаковы окно, масштабирование, `center=True` и длина результата.
Перед Griffin–Lim обращается степенное сжатие амплитуды.
По умолчанию используется **обычный Griffin–Lim без momentum**, 60 итераций, инициализация исходной фазой.
Это уточнение важно: алгоритм обновляет фазу, но использует хорошую начальную оценку.
Для случайного старта выберите `GL_INITIAL_PHASE = "random"` до эксперимента.
Метрики вычисляются на float-сигналах без clipping и индивидуального выравнивания RMS.


In [6]:
def sync_device():
    if device.type == "cuda":
        torch.cuda.synchronize(device)

def stft(wave):
    return torch.stft(wave, n_fft=h.n_fft, hop_length=h.hop_size, win_length=h.win_size,
                      window=WINDOW, center=True, pad_mode="reflect", normalized=False,
                      onesided=True, return_complex=True)

def istft(spectrum, length):
    return torch.istft(spectrum, n_fft=h.n_fft, hop_length=h.hop_size, win_length=h.win_size,
                       window=WINDOW, center=True, normalized=False, onesided=True, length=length)

@torch.inference_mode()
def griffin_lim(magnitude, initial_phase, length, iterations):
    phase = initial_phase.clone()
    for _ in range(iterations):
        wave = istft(torch.polar(magnitude, phase), length)
        rebuilt = stft(wave)
        if rebuilt.shape != magnitude.shape:
            raise RuntimeError(f"Несовпадение размеров STFT: {rebuilt.shape} != {magnitude.shape}")
        phase = torch.angle(rebuilt)
    return istft(torch.polar(magnitude, phase), length)

def read_wave(path):
    wave, source_sr = sf.read(path, dtype="float32", always_2d=True)
    wave = wave.mean(axis=1)
    if not np.isfinite(wave).all() or len(wave) == 0:
        raise ValueError(f"Пустое или нечисловое аудио: {path}")
    if source_sr != SR:
        divisor = math.gcd(int(source_sr), SR)
        wave = resample_poly(wave, SR // divisor, int(source_sr) // divisor).astype(np.float32)
    return np.ascontiguousarray(wave), int(source_sr)

@torch.inference_mode()
def enhance_three(noisy_wave, file_seed):
    raw = torch.as_tensor(noisy_wave, dtype=torch.float32, device=device).unsqueeze(0)
    length = raw.shape[-1]
    if length <= h.n_fft // 2:
        raise ValueError("Запись слишком короткая для reflect padding STFT.")
    energy = raw.square().sum()
    if not torch.isfinite(energy) or energy.item() <= 1e-12:
        raise ValueError("Зашумлённый сигнал пустой или практически без энергии.")
    factor = torch.sqrt(length / energy)   # Только noisy, как в официальном inference.

    sync_device()
    started = time.perf_counter()
    spectrum = stft(raw * factor)
    # Эти стабилизаторы повторяют подготовку входов авторов.
    model_magnitude = (spectrum.real.square() + spectrum.imag.square() + 1e-9).sqrt().pow(h.compress_factor)
    model_phase = torch.atan2(spectrum.imag + 1e-10, spectrum.real + 1e-5)
    magnitude_c, predicted_phase, _ = model(model_magnitude, model_phase)
    magnitude = magnitude_c.clamp_min(0).pow(1.0 / h.compress_factor)
    if not torch.isfinite(magnitude).all() or not torch.isfinite(predicted_phase).all():
        raise FloatingPointError("Модель выдала NaN/Inf.")
    noisy_phase = torch.angle(spectrum)   # Точная исходная фаза для варианта A.
    sync_device()
    shared_seconds = time.perf_counter() - started

    if GL_INITIAL_PHASE == "noisy":
        initial_phase = noisy_phase
    else:
        generator = torch.Generator(device="cpu").manual_seed(int(file_seed))
        initial_phase = (torch.rand(magnitude.shape, generator=generator) * (2 * math.pi) - math.pi).to(device)

    reconstructions = {
        "original_phase": lambda: istft(torch.polar(magnitude, noisy_phase), length),
        "griffin_lim": lambda: griffin_lim(magnitude, initial_phase, length, GL_ITERATIONS),
        "mp_senet": lambda: istft(torch.polar(magnitude, predicted_phase), length),
    }
    waves = {"noisy": np.asarray(noisy_wave, dtype=np.float32)}
    timings = {}
    magnitude_snapshot = magnitude.clone()
    for method, reconstruct in reconstructions.items():
        sync_device()
        started = time.perf_counter()
        wave = reconstruct()
        sync_device()
        seconds = time.perf_counter() - started
        if wave.shape[-1] != length or not torch.isfinite(wave).all():
            raise FloatingPointError(f"Некорректный результат {method}")
        waves[method] = (wave / factor).squeeze(0).cpu().numpy()
        timings[method] = {"shared_forward_s": shared_seconds, "reconstruction_s": seconds,
                           "pipeline_s": shared_seconds + seconds,
                           "RTF": (shared_seconds + seconds) / (length / SR)}
    assert torch.equal(magnitude, magnitude_snapshot), "Общая амплитуда была изменена"
    return waves, timings


## 5. Проверки до работы с датасетом
Проверяются обратимость STFT, обращение сжатия и одна итерация Griffin–Lim при уже правильной фазе.
Это проверки вычислений; они не заменяют оценку на речи.


In [7]:
with torch.inference_mode():
    generator = torch.Generator(device="cpu").manual_seed(SEED)
    probe = (torch.randn(1, SR, generator=generator) * 0.05).to(device)
    spec = stft(probe)
    back = istft(spec, probe.shape[-1])
    error = (back - probe).abs().max().item()
    assert error < 2e-5, error
    restored_mag = spec.abs().pow(h.compress_factor).pow(1.0 / h.compress_factor)
    assert torch.allclose(restored_mag, spec.abs(), rtol=2e-5, atol=2e-5)
    gl_back = griffin_lim(spec.abs(), torch.angle(spec), probe.shape[-1], iterations=1)
    assert torch.allclose(gl_back, probe, rtol=2e-4, atol=2e-5)
print(f"Проверки STFT / compression / Griffin–Lim пройдены. Max error={error:.2e}")


Проверки STFT / compression / Griffin–Lim пройдены. Max error=3.73e-08


## 6. Метрики
PESQ-WB, ESTOI и SI-SDR требуют чистую запись. Она используется **только здесь**.
Ошибки метрик сохраняются явно; NaN не превращаются в нули. В сводке показано число успешных измерений.
DNSMOS из старого ноутбука не используется. Задержка аудио автоматически не подгоняется: это изменило бы протокол сравнения.


In [8]:
def si_sdr(reference, estimate):
    ref = np.asarray(reference, dtype=np.float64)
    est = np.asarray(estimate, dtype=np.float64)
    ref = ref - ref.mean()
    est = est - est.mean()
    ref_energy = np.dot(ref, ref)
    if ref_energy <= 1e-12:
        raise ValueError("SI-SDR не определена для беззвучного эталона")
    projection = ref * (np.dot(est, ref) / ref_energy)
    residual = est - projection
    if np.dot(projection, projection) <= 1e-20:
        raise ValueError("В оценке нет измеримой проекции на эталон")
    return float(10 * np.log10(np.dot(projection, projection) / max(np.dot(residual, residual), 1e-20)))

def evaluate_wave(reference, estimate):
    reference = np.asarray(reference, dtype=np.float64)
    estimate = np.asarray(estimate, dtype=np.float64)
    if reference.shape != estimate.shape or not np.isfinite(estimate).all():
        raise ValueError("Длины должны совпадать; NaN/Inf недопустимы")
    results = {}
    functions = {
        "PESQ": lambda: pesq(SR, reference, estimate, "wb"),
        "ESTOI": lambda: stoi(reference, estimate, SR, extended=True),
        "SI_SDR": lambda: si_sdr(reference, estimate),
    }
    for name, calculate in functions.items():
        try:
            value = float(calculate())
            if not math.isfinite(value):
                raise ValueError("Метрика вернула NaN/Inf")
            results[name] = value
            results[name + "_error"] = ""
        except Exception as exc:
            results[name] = float("nan")
            results[name + "_error"] = f"{type(exc).__name__}: {exc}"
    return results

# Проверяем в том числе нечувствительность SI-SDR к постоянному усилению.
probe_np = probe.squeeze(0).cpu().numpy()
assert abs(si_sdr(probe_np, probe_np + np.roll(probe_np, 1) * 0.1)
           - si_sdr(probe_np, (probe_np + np.roll(probe_np, 1) * 0.1) * 3)) < 1e-5
print("Функции метрик готовы.")


Функции метрик готовы.


In [9]:
if not NOISY_DIR.is_dir() or not CLEAN_DIR.is_dir():
    raise FileNotFoundError("Укажите существующие NOISY_DIR и CLEAN_DIR в ячейке 2.")
noisy_files = {p.name: p for p in NOISY_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".wav"}
clean_files = {p.name: p for p in CLEAN_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".wav"}
names = sorted(set(noisy_files) & set(clean_files))
if not names:
    raise RuntimeError("Не найдено WAV-пар с одинаковыми именами.")
missing_clean = sorted(set(noisy_files) - set(clean_files))
missing_noisy = sorted(set(clean_files) - set(noisy_files))
print(f"Пар: {len(names)}; без clean: {len(missing_clean)}; без noisy: {len(missing_noisy)}")
if missing_clean or missing_noisy:
    warnings.warn("Набор неполный. Списки пропущенных пар будут записаны в manifest.")
if N_FILES is not None and N_FILES < len(names):
    names = sorted(random.Random(SEED).sample(names, N_FILES))
print("Файлов в этом запуске:", len(names))

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(), "revision": REVISION,
    "checkpoint": "best_ckpt/g_best_vb", "sha256": SOURCE_HASHES,
    "model_config": vars(h), "device": str(device), "platform": platform.platform(),
    "device_name": torch.cuda.get_device_name(device) if device.type == "cuda" else platform.processor(),
    "python": sys.version, "seed": SEED, "gl_iterations": GL_ITERATIONS,
    "gl_initial_phase": GL_INITIAL_PHASE, "noisy_dir": str(NOISY_DIR), "clean_dir": str(CLEAN_DIR),
    "files": names, "missing_clean": missing_clean, "missing_noisy": missing_noisy,
    "n_files_requested": N_FILES, "bootstrap_repeats": BOOTSTRAP_REPEATS,
    "packages": {name: importlib.metadata.version(name) for name in
                 ["torch", "numpy", "scipy", "pandas", "soundfile", "pesq", "pystoi"]},
    "protocol": "same predicted magnitude; native checkpoint semantics; no clipping or per-method RMS correction",
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2))


FileNotFoundError: Укажите существующие NOISY_DIR и CLEAN_DIR в ячейке 2.

## 7. Обработка и сохранение CSV
Прогрев модели исключён из времени. Затем каждая запись обрабатывается целиком, batch size = 1.
Сохраняются `metrics_per_file.csv` и `errors.csv`, включая промежуточное состояние после каждого файла.
Несовпадение длины или sample rate пары считается ошибкой, а не скрывается обрезкой.
Аудиопримеры сохраняются как float WAV без clipping; playback gain будет общим для всех методов.
RTF включает общий forward MP-SENet и реконструкцию, исключает чтение, метрики и запись файлов.
Это время данного эксперимента: forward вычисляет обе головы даже для A/B.


In [ ]:
with torch.inference_mode():
    # Короткий прогрев нейросети: не входит в измерения.
    warm = torch.randn(1, SR, device=device) * 0.03
    warm_spec = stft(warm)
    model((warm_spec.abs().square() + 1e-9).sqrt().pow(h.compress_factor),
          torch.atan2(warm_spec.imag + 1e-10, warm_spec.real + 1e-5))
    sync_device()

records, errors, audio_examples = [], [], []
for index, name in enumerate(tqdm(names, desc="Сравнение фаз")):
    try:
        noisy, noisy_sr = read_wave(noisy_files[name])
        clean, clean_sr = read_wave(clean_files[name])
        if noisy_sr != clean_sr or noisy.shape != clean.shape:
            raise ValueError(f"Пара не согласована: sr={noisy_sr}/{clean_sr}, length={len(noisy)}/{len(clean)}")
        if np.mean(clean.astype(np.float64) ** 2) <= 1e-12:
            raise ValueError("Беззвучный чистый эталон")
        file_seed = (SEED + int(hashlib.sha256(name.encode()).hexdigest()[:8], 16)) % (2 ** 32)
        waves, timings = enhance_three(noisy, file_seed)
        file_records = []
        hashes = {"noisy_sha256": file_sha256(noisy_files[name]), "clean_sha256": file_sha256(clean_files[name])}
        for method in METHODS:
            timing = timings.get(method, {"shared_forward_s": 0.0, "reconstruction_s": 0.0, "pipeline_s": 0.0, "RTF": 0.0})
            wave = waves[method]
            file_records.append({"file": name, "method": method, "duration_s": len(clean) / SR,
                                 "input_sr": noisy_sr, "peak": float(np.max(np.abs(wave))),
                                 "fraction_over_1": float(np.mean(np.abs(wave) > 1)),
                                 **hashes, **evaluate_wave(clean, wave), **timing})
        records.extend(file_records)
        if len(audio_examples) < SAVE_AUDIO_EXAMPLES:
            example_dir = RUN_DIR / "audio" / Path(name).stem
            example_dir.mkdir(parents=True, exist_ok=True)
            for label, wave in {"clean": clean, **waves}.items():
                sf.write(example_dir / f"{label}.wav", wave, SR, subtype="FLOAT")
            audio_examples.append(example_dir)
    except Exception as exc:
        errors.append({"file": name, "error": f"{type(exc).__name__}: {exc}"})
        tqdm.write(f"Ошибка {name}: {exc}")
        if device.type == "cuda":
            torch.cuda.empty_cache()
    pd.DataFrame(records).to_csv(RUN_DIR / "metrics_per_file.csv", index=False)
    pd.DataFrame(errors, columns=["file", "error"]).to_csv(RUN_DIR / "errors.csv", index=False)

df = pd.DataFrame(records)
if df.empty:
    raise RuntimeError(f"Нет результатов. Проверьте {RUN_DIR / 'errors.csv'}")
print(f"Обработано: {df['file'].nunique()}/{len(names)}; ошибок файлов: {len(errors)}")
metric_failures = {metric: int(df[metric].isna().sum()) for metric in METRICS}
print("Ошибки метрик:", metric_failures)
if errors or any(metric_failures.values()):
    warnings.warn("Есть пропуски. Проверяйте n в сводке и пары файлов перед интерпретацией.")


## 8. Сводка и парные сравнения
Средние/медианы считаются по **одним и тем же успешным файлам всех четырёх методов для каждой метрики**.
В парных сравнениях также используются только общие конечные значения. Положительная разность означает преимущество первого метода.
95% bootstrap-интервалы считаются по файлам; корреляция записей одного диктора не учтена.
Метрики не заменяют прослушивание и не доказывают субъективное превосходство.


In [ ]:
summary_rows, paired_rows = [], []
contrasts = [("original_phase", "noisy"), ("griffin_lim", "original_phase"),
             ("mp_senet", "original_phase"), ("mp_senet", "griffin_lim")]
rng = np.random.default_rng(SEED)
for metric in METRICS:
    pivot = df.pivot(index="file", columns="method", values=metric).reindex(columns=METHODS)
    common = pivot.replace([np.inf, -np.inf], np.nan).dropna()
    for method in METHODS:
        values = common[method]
        summary_rows.append({"metric": metric, "method": method, "n_common": len(values),
                             "n_available": int(df.loc[df.method == method, metric].notna().sum()),
                             "mean": values.mean(), "median": values.median(), "std": values.std()})
    for first, second in contrasts:
        delta = (common[first] - common[second]).to_numpy()
        low, high = float("nan"), float("nan")
        if len(delta) > 1:
            bootstrap_means = np.array([rng.choice(delta, size=len(delta), replace=True).mean()
                                        for _ in range(BOOTSTRAP_REPEATS)])
            low, high = np.quantile(bootstrap_means, [0.025, 0.975])
        paired_rows.append({"metric": metric, "first": first, "second": second, "n": len(delta),
                            "mean_difference": delta.mean() if len(delta) else float("nan"),
                            "ci95_low": low, "ci95_high": high})
summary = pd.DataFrame(summary_rows)
paired = pd.DataFrame(paired_rows)
summary.to_csv(RUN_DIR / "summary.csv", index=False)
paired.to_csv(RUN_DIR / "paired_differences.csv", index=False)
display(summary.round(4))
display(paired.round(4))

timing = df[df.method != "noisy"].groupby("method").agg(
    n=("file", "count"), total_seconds=("pipeline_s", "sum"), duration_seconds=("duration_s", "sum"),
    mean_reconstruction_s=("reconstruction_s", "mean"), median_rtf=("RTF", "median"))
timing["aggregate_rtf"] = timing.total_seconds / timing.duration_seconds
timing.to_csv(RUN_DIR / "timing.csv")
display(timing.round(4))
print("RTF < 1: обработка быстрее длительности записи на данном устройстве.")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ["#888888", "#4c72b0", "#dd8452", "#55a868"]
for ax, metric in zip(axes, METRICS):
    values = summary[summary.metric == metric].set_index("method").reindex(METHODS)["mean"]
    ax.bar(METHODS, values, color=colors)
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.25)
fig.suptitle("Одинаковая предсказанная амплитуда — разные способы восстановления фазы")
fig.tight_layout()
fig.savefig(RUN_DIR / "comparison.png", dpi=160, bbox_inches="tight")
plt.show()


## 9. Прослушивание
Для каждой записи используется **один общий gain**, рассчитанный по пику всех пяти вариантов.
`normalize=False` отключает индивидуальную нормализацию плеера. Метрики и float WAV при этом не меняются.
Слушайте несколько примеров: качество речи, остаточный шум, металлические артефакты и громкость.


In [ ]:
for example_dir in audio_examples:
    signals = {label: sf.read(example_dir / f"{label}.wav", dtype="float32")[0]
               for label in ["clean", *METHODS]}
    common_peak = max(float(np.max(np.abs(wave))) for wave in signals.values())
    gain = 0.95 / max(1.0, common_peak)
    print(example_dir.name, "| общий playback gain:", round(gain, 4))
    for label, wave in signals.items():
        print(label)
        display(Audio(wave * gain, rate=SR, normalize=False))
print("Все результаты:", RUN_DIR)


## Как читать результат
- A против B: эффект итерационного восстановления фазы при одной амплитуде.
- A против C: эффект предсказанной фазы MP-SENet при той же амплитуде.
- B против C: сравнение Griffin–Lim с фазовой головой готового MP-SENet.
- Baseline показывает, улучшает ли обработка качество относительно входа.
- Качество и скорость относятся к этому checkpoint, выбранной инициализации GL и указанным файлам.
- Уменьшение SI-SDR при хорошем PESQ возможно из-за изменения временной структуры; проверяйте сигнал и прослушивание.
- Смена `GL_INITIAL_PHASE` — отдельный запуск с отдельным manifest. Не выбирайте лучший вариант по итоговому test.

Файлы: `manifest.json`, `metrics_per_file.csv`, `errors.csv`, `summary.csv`,
`paired_differences.csv`, `timing.csv`, `comparison.png`, `audio/`.
